<a href="https://colab.research.google.com/github/mjahangiralam/AdvancedMacroeconomics/blob/main/Dynare%2BGnu_Octave_Solow_Simulation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Solow Growth Model using Dynare and GNU Octave

This notebook demonstrates how to simulate the Solow growth model using Dynare within a computational environment.

The objective is to connect:
- macroeconomic theory
- numerical simulation

We focus on how the economy transitions between steady states after a structural change.

## Step 1: Setting up the Environment

We install the necessary tools:

- GNU Octave: numerical computing environment (similar to MATLAB)
- Dynare: specialized software for solving macroeconomic models

This allows us to run `.mod` files that describe dynamic economic systems.

In [ ]:
!apt-get update -qq
!apt-get install -y octave dynare

## Step 2: Solow Growth Model

We now implement the Solow growth model, a fundamental framework in macroeconomics.

### Production Function
$$ y_t = A k_{t-1}^{\alpha} $$

- Output depends on capital
- $A$ captures productivity

---

### Capital Accumulation
$$ k_t = \frac{s y_t + (1-\delta)k_{t-1}}{(1+n)(1+g)} $$

- Savings determines investment
- Depreciation reduces capital
- Population and technology dilute capital

---

### Key Insight

Due to diminishing returns:

> The economy converges to a steady state.  
> Higher savings raises output levels, but not long-run growth.

In [ ]:
%%bash

cat <<EOF > solow.mod
var y k c w r sav;
varexo a x z;

parameters alpha delta n g s;

alpha = 0.333;
delta = 0.03;
n = 0.01;
g = 0.02;
s = 0.30;

model;
y = a*(k(-1)^alpha);
c = (1-(s*x))*y;
k = (1/((1+n*z)*(1+g)))*((s*x*y) + (1-delta)*k(-1));
r = alpha*a*k(-1)^(alpha-1) - delta;
w = (1-alpha)*a*k(-1)^alpha;
sav = s*x;
end;

initval;
k = 5.3;
c = 1.38;
y = 1.7;
a = 1;
r = 0.075;
w = 1.16;
x = 1;
z = 1;
end;

steady;

endval;
z = 1.01;
end;

steady;
check;

perfect_foresight_setup(periods=100);
perfect_foresight_solver;

save('solow_results.mat', 'oo_');
EOF

octave --eval "addpath('/usr/lib/dynare/matlab'); dynare solow.mod"

## Step 3: Transition Dynamics

We introduce a structural change using:
`endval`

Dynare computes how the economy transitions over time.

### Interpretation
- The economy starts in steady state
- A shock shifts the system
- Adjustment occurs gradually over time

This reflects real-world capital accumulation processes.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt


def simulate_solow_saving_shock(
    T: int = 81,
    A: float = 1.0,
    alpha: float = 0.333,
    s0: float = 0.30,
    s1: float = 0.303,
    n: float = 0.01,
    g: float = 0.02,
    delta: float = 0.03,
):
    """
    Simulate Solow model after a permanent saving-rate shock.
    """

    t = np.arange(T)

    # Saving path (shock at t=1)
    s_path = np.full(T, s0)
    s_path[1:] = s1

    # Initial steady state
    denom = (1 + n) * (1 + g) - (1 - delta)
    if denom <= 0:
        raise ValueError("Invalid parameters.")

    k_ss0 = (s0 * A / denom) ** (1.0 / (1.0 - alpha))

    # Storage
    k = np.empty(T)
    y = np.empty(T)
    c = np.empty(T)
    w = np.empty(T)
    r = np.empty(T)

    k[0] = k_ss0

    # Simulation
    for i in range(T):
        y[i] = A * k[i] ** alpha
        c[i] = (1.0 - s_path[i]) * y[i]
        w[i] = (1.0 - alpha) * y[i]
        r[i] = alpha * A * k[i] ** (alpha - 1.0) - delta

        if i < T - 1:
            k[i + 1] = (
                s_path[i] * y[i] + (1.0 - delta) * k[i]
            ) / ((1.0 + n) * (1.0 + g))

    return t, k, y, c, r, w, s_path


def make_figure():
    t, k, y, c, r, w, s_path = simulate_solow_saving_shock()

    fig, axes = plt.subplots(3, 2, figsize=(12, 8))
    axes = axes.ravel()

    series = [
        (k, "Capital"),
        (y, "Output"),
        (c, "Consumption"),
        (r, "Real Interest Rate"),
        (w, "Real Wage"),
        (s_path, "Savings Rate"),
    ]

    for ax, (data, title) in zip(axes, series):
        ax.plot(t, data, linewidth=2)

        # Titles and labels (FIXED)
        ax.set_title(title, fontsize=13, fontweight="bold")
        ax.set_xlabel("Time")
        ax.set_ylabel(title)

        # Visual improvements
        ax.set_xlim(0, 80)
        ax.grid(True, alpha=0.3)

        # Shock marker at t=1
        ax.axvline(x=1, linestyle="--", alpha=0.5)

    plt.tight_layout()
    plt.show()


if __name__ == "__main__":
    make_figure()

## Step 4: Interpreting Results

The simulation highlights the main predictions of the Solow model:

- Capital increases gradually
- Output and wages rise with capital
- Consumption adjusts with savings behavior
- Interest rates decline due to diminishing returns

---

### Core Result

> Changes in saving affect income levels,  
> but not long-run growth rates.

The economy converges smoothly to a new steady state.